In [1]:
!pip install pymilvus[milvus_lite] -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 33.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.2/315.2 kB 15.6 MB/s eta 0:00:00


In [2]:
# HuggingFace login via Kaggle Secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)
print("HuggingFace login successful")

HuggingFace login successful


In [3]:
import os
import json
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient

BASE = "/kaggle/input/datasets/tanuadhikari/retrieval"

TRAIN_DOC_LABELS_PATH  = f"{BASE}/train_legal_docs.txt"
TRAIN_QUERY_LABELS_PATH= f"{BASE}/train_queries.txt"

VAL_DOC_LABELS_PATH    = f"{BASE}/val_legal_docs.txt"
VAL_QUERY_LABELS_PATH  = f"{BASE}/val_queries.txt"

TEST_DOC_LABELS_PATH   = f"{BASE}/test_legal_docs.txt"
TEST_QUERY_LABELS_PATH = f"{BASE}/test_queries.txt"

# Where Milvus DB will be saved
DB_PATH = "/kaggle/working/legal_ir.db"
COLLECTION_NAME = "legal_corpus"

print("Paths configured")

Paths configured


In [4]:
def load_label_file(path):
    labels = {}
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            doc_id, labels_str = parts
            labels[doc_id] = labels_str.split(",")
    print(f"  Loaded labels for {len(labels)} documents from {os.path.basename(path)}")
    return labels

print("Loading all label files...")
train_doc_labels   = load_label_file(TRAIN_DOC_LABELS_PATH)
train_query_labels = load_label_file(TRAIN_QUERY_LABELS_PATH)
val_doc_labels     = load_label_file(VAL_DOC_LABELS_PATH)
val_query_labels   = load_label_file(VAL_QUERY_LABELS_PATH)
test_doc_labels    = load_label_file(TEST_DOC_LABELS_PATH)
test_query_labels  = load_label_file(TEST_QUERY_LABELS_PATH)
print("All label files loaded!")

Loading all label files...
  Loaded labels for 4320 documents from train_legal_docs.txt
  Loaded labels for 827 documents from train_queries.txt
  Loaded labels for 1023 documents from val_legal_docs.txt
  Loaded labels for 118 documents from val_queries.txt
  Loaded labels for 1727 documents from test_legal_docs.txt
  Loaded labels for 237 documents from test_queries.txt
All label files loaded!


In [5]:
print("Loading IL-TUR PCR dataset from HuggingFace...")
dataset = load_dataset("Exploration-Lab/IL-TUR", "pcr")
print("Splits available:", list(dataset.keys()))

def extract_sentences(split):
    """
    From a dataset split, build a dict: {str(doc_id): [sentence1, sentence2, ...]}
    """
    result = {}
    for sample in split:
        result[str(sample['id'])] = sample['text']  # 'text' is a list of sentences
    return result

# Corpus documents (candidates)
train_doc_sentences = extract_sentences(dataset['train_candidates'])
val_doc_sentences   = extract_sentences(dataset['dev_candidates'])
test_doc_sentences  = extract_sentences(dataset['test_candidates'])

# Query documents
train_query_sentences = extract_sentences(dataset['train_queries'])
val_query_sentences   = extract_sentences(dataset['dev_queries'])
test_query_sentences  = extract_sentences(dataset['test_queries'])

print(f"Train corpus docs: {len(train_doc_sentences)}")
print(f"Val corpus docs:   {len(val_doc_sentences)}")
print(f"Test corpus docs:  {len(test_doc_sentences)}")
print(f"Train queries:     {len(train_query_sentences)}")
print(f"Val queries:       {len(val_query_sentences)}")
print(f"Test queries:      {len(test_query_sentences)}")

Loading IL-TUR PCR dataset from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

pcr/train_candidates-00000-of-00001.parq(…):   0%|          | 0.00/77.0M [00:00<?, ?B/s]

pcr/dev_candidates-00000-of-00001.parque(…):   0%|          | 0.00/25.1M [00:00<?, ?B/s]

pcr/test_candidates-00000-of-00001.parqu(…):   0%|          | 0.00/34.2M [00:00<?, ?B/s]

...

Generating test_candidates split:   0%|          | 0/1727 [00:00<?, ? examples/s]

Generating train_queries split:   0%|          | 0/827 [00:00<?, ? examples/s]

Generating dev_queries split:   0%|          | 0/118 [00:00<?, ? examples/s]

Generating test_queries split:   0%|          | 0/237 [00:00<?, ? examples/s]

Splits available: ['train_candidates', 'dev_candidates', 'test_candidates', 'train_queries', 'dev_queries', 'test_queries']
Train corpus docs: 4320
Val corpus docs:   1023
Test corpus docs:  1727
Train queries:     827
Val queries:       118
Test queries:      237


In [6]:
# Best config from Table 2 of the paper
CORPUS_ROLES = {
    "Facts",
    "Issues",
    "Ratio of the decision",   # = Reasoning in paper terminology
}

# Query: simulate limited info scenario (Facts + Issues only)
QUERY_ROLES = {
    "Facts",
    "Issues",
}

def filter_by_roles(doc_sentences_dict, doc_labels_dict, keep_roles, split_name):
    filtered = {}
    skipped = 0
    for doc_id, labels in doc_labels_dict.items():
        sents = doc_sentences_dict.get(doc_id)
        if sents is None:
            skipped += 1
            continue
        # Align length — label file and sentence list may differ slightly
        min_len = min(len(sents), len(labels))
        kept = [
            s.strip()
            for s, l in zip(sents[:min_len], labels[:min_len])
            if l in keep_roles
        ]
        if kept:
            filtered[doc_id] = " ".join(kept)
    print(f"[{split_name}] Filtered: {len(filtered)} docs | Skipped (not in HF): {skipped}")
    return filtered

print("=== Filtering CORPUS documents (Facts + Issues + Reasoning) ===")
train_corpus = filter_by_roles(train_doc_sentences, train_doc_labels, CORPUS_ROLES, "train_corpus")
val_corpus   = filter_by_roles(val_doc_sentences,   val_doc_labels,   CORPUS_ROLES, "val_corpus")
test_corpus  = filter_by_roles(test_doc_sentences,  test_doc_labels,  CORPUS_ROLES, "test_corpus")

print("\n=== Filtering QUERY documents (Facts + Issues only) ===")
train_queries = filter_by_roles(train_query_sentences, train_query_labels, QUERY_ROLES, "train_queries")
val_queries   = filter_by_roles(val_query_sentences,   val_query_labels,   QUERY_ROLES, "val_queries")
test_queries  = filter_by_roles(test_query_sentences,  test_query_labels,  QUERY_ROLES, "test_queries")

=== Filtering CORPUS documents (Facts + Issues + Reasoning) ===
[train_corpus] Filtered: 4056 docs | Skipped (not in HF): 0
[val_corpus] Filtered: 1001 docs | Skipped (not in HF): 0
[test_corpus] Filtered: 1705 docs | Skipped (not in HF): 0

=== Filtering QUERY documents (Facts + Issues only) ===
[train_queries] Filtered: 685 docs | Skipped (not in HF): 0
[val_queries] Filtered: 101 docs | Skipped (not in HF): 0
[test_queries] Filtered: 192 docs | Skipped (not in HF): 0


In [8]:
!pip install xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.5 MB/s eta 0:00:00:00:01


In [8]:
print("Loading embedding model")
embed_model = SentenceTransformer("Snowflake/snowflake-arctic-embed-l")
DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {DIM}")

Loading embedding model


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Embedding dimension: 1024


Paper specifies:
- IVF-FLAT index
- nlist = 2048
- L2 distance
- Text stored up to 60,000 characters

In [9]:
from pymilvus import MilvusClient  # just this, nothing else needed

# Connect
client = MilvusClient(DB_PATH)
print("Milvus Lite connected at", DB_PATH)

# Drop if exists
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)
    print(f"Dropped existing collection '{COLLECTION_NAME}'")

# Create collection
client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=DIM,
    metric_type="L2",
    id_field_name="id",
    vector_field_name="embedding",
    enable_dynamic_field=True,
)
print(f"Collection '{COLLECTION_NAME}' created with dim={DIM}")

# Build index params
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="embedding",
    index_type="IVF_FLAT",
    metric_type="L2",
    params={"nlist": 2048}
)

client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)
print("IVF-FLAT index created with nlist=2048")

# Load
client.load_collection(COLLECTION_NAME)
print("Collection loaded — ready for insert & search")

Milvus Lite connected at /kaggle/working/legal_ir.db
Collection 'legal_corpus' created with dim=1024
IVF-FLAT index created with nlist=2048
Collection loaded — ready for insert & search


In [10]:
# Merge all corpus splits into one unified corpus dict
# (The retrieval database = all legal documents across all splits)
full_corpus = {}
full_corpus.update(train_corpus)
full_corpus.update(val_corpus)
full_corpus.update(test_corpus)
print(f"Total corpus size: {len(full_corpus)} documents")

BATCH_SIZE = 64
corpus_ids   = list(full_corpus.keys())
corpus_texts = [full_corpus[d] for d in corpus_ids]

print(f"Encoding and inserting {len(corpus_texts)} documents in batches of {BATCH_SIZE}...")

for batch_start in range(0, len(corpus_texts), BATCH_SIZE):
    batch_texts = corpus_texts[batch_start : batch_start + BATCH_SIZE]
    batch_ids   = corpus_ids  [batch_start : batch_start + BATCH_SIZE]

    # Encode batch — normalize_embeddings=True makes L2 ≈ cosine similarity
    embeddings = embed_model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
    ).tolist()

    # Build records for Milvus
    data = [
        {
            # Primary key: convert doc_id string to int for Milvus
            "id":        int(bid) if bid.isdigit() else abs(hash(bid)) % (2**31),
            "embedding": emb,
            "doc_id":    bid,              # original string ID stored as dynamic field
            "text":      txt[:60000],      # paper says 60,000 chars (was 4000 — fixed!)
        }
        for bid, emb, txt in zip(batch_ids, embeddings, batch_texts)
    ]

    client.insert(collection_name=COLLECTION_NAME, data=data)

    # Print progress every 10 batches
    if (batch_start // BATCH_SIZE) % 10 == 0:
        inserted_so_far = min(batch_start + BATCH_SIZE, len(corpus_texts))
        print(f"  Inserted {inserted_so_far}/{len(corpus_texts)}")

print("All corpus documents inserted into Milvus!")

# Save full corpus texts to disk for BM25 (next stage needs raw text)
with open("/kaggle/working/full_corpus.json", "w") as f:
    json.dump(full_corpus, f)
print("Saved full_corpus.json — will be used by BM25 stage")

Total corpus size: 5569 documents
Encoding and inserting 5569 documents in batches of 64...
  Inserted 64/5569
  Inserted 704/5569
  Inserted 1344/5569
  Inserted 1984/5569
  Inserted 2624/5569
  Inserted 3264/5569
  Inserted 3904/5569
  Inserted 4544/5569
  Inserted 5184/5569
All corpus documents inserted into Milvus!
Saved full_corpus.json — will be used by BM25 stage


In [11]:
def search_vector_db(query_text, limit=1000, nprobe=64):
    # Encode query with same model as corpus
    query_vec = embed_model.encode(
        [query_text],
        normalize_embeddings=True
    ).tolist()

    # Search Milvus with nprobe — this was missing before!
    results = client.search(
        collection_name=COLLECTION_NAME,
        data=query_vec,
        limit=limit,
        output_fields=["doc_id"],
        search_params={
            "metric_type": "L2",
            "params": {"nprobe": nprobe}   # paper says nprobe controls search depth
        }
    )

    # Parse results into a clean list
    hits = [
        {"doc_id": hit["entity"]["doc_id"], "distance": hit["distance"]}
        for hit in results[0]
    ]
    return hits

print("search_vector_db() function defined")

search_vector_db() function defined


In [12]:
# Pick first train query for sanity check
sample_qid   = list(train_queries.keys())[0]
sample_qtext = train_queries[sample_qid]

print(f"Query ID: {sample_qid}")
print(f"Query text preview: {sample_qtext[:300]}")
print()

hits = search_vector_db(sample_qtext, limit=10)
print("Top-10 retrieved candidates:")
for i, h in enumerate(hits):
    print(f"  Rank {i+1}: doc_id={h['doc_id']}  L2_distance={h['distance']:.4f}")

Query ID: 0000232986
Query text preview: PETITIONER: <ORG>, MADRAS Vs. RESPONDENT: COLLECTOR OF MADRAS DATE OF JUDGMENT01/05/1975 BENCH: <NAME>, V.R. BENCH: <NAME>, V.R. SARKARIA, RANJIT SINGH GUPTA, A.C. CITATION: 1975 AIR 1670 1975 SCR 403 CITATOR INFO : R 1977 SC 580 (9) R 1989 SC1222 (5) RF 1992 SC 666 (3) RF 1992 SC1406 (14) ACT: Land

Top-10 retrieved candidates:
  Rank 1: doc_id=0000232986  L2_distance=0.0000
  Rank 2: doc_id=0001491022  L2_distance=0.5081
  Rank 3: doc_id=0001329646  L2_distance=0.5639
  Rank 4: doc_id=0001497108  L2_distance=0.5816
  Rank 5: doc_id=0000505842  L2_distance=0.5963
  Rank 6: doc_id=0001006731  L2_distance=0.6082
  Rank 7: doc_id=0000975797  L2_distance=0.6088
  Rank 8: doc_id=0001559215  L2_distance=0.6108
  Rank 9: doc_id=0000955281  L2_distance=0.6139
  Rank 10: doc_id=0001239855  L2_distance=0.6239


E0410 20:35:47.066963      55 chttp2_transport.cc:1385] unix:/tmp/tmp0uoy09bt_legal_ir.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [13]:
def run_all_queries(queries_dict, split_name, limit=7, nprobe=64):
    all_results = {}
    total = len(queries_dict)

    for i, (qid, qtext) in enumerate(queries_dict.items()):
        hits = search_vector_db(qtext, limit=limit, nprobe=nprobe)

        # Add rank field (1-indexed)
        all_results[qid] = [
            {"doc_id": h["doc_id"], "distance": h["distance"], "rank": r+1}
            for r, h in enumerate(hits)
        ]

        # Progress log every 50 queries
        if (i + 1) % 50 == 0 or (i + 1) == total:
            print(f"  [{split_name}] Processed {i+1}/{total} queries")

    return all_results


print("=== Running TRAIN queries ===")
train_vdb_results = run_all_queries(train_queries, "train")

print("\n=== Running VAL queries ===")
val_vdb_results   = run_all_queries(val_queries, "val")

print("\n=== Running TEST queries ===")
test_vdb_results  = run_all_queries(test_queries, "test")

print("\nAll queries done!")

=== Running TRAIN queries ===
  [train] Processed 50/685 queries
  [train] Processed 100/685 queries
  [train] Processed 150/685 queries
  [train] Processed 200/685 queries
  [train] Processed 250/685 queries
  [train] Processed 300/685 queries
  [train] Processed 350/685 queries
  [train] Processed 400/685 queries
  [train] Processed 450/685 queries
  [train] Processed 500/685 queries
  [train] Processed 550/685 queries
  [train] Processed 600/685 queries
  [train] Processed 650/685 queries
  [train] Processed 685/685 queries

=== Running VAL queries ===
  [val] Processed 50/101 queries
  [val] Processed 100/101 queries
  [val] Processed 101/101 queries

=== Running TEST queries ===
  [test] Processed 50/192 queries
  [test] Processed 100/192 queries
  [test] Processed 150/192 queries
  [test] Processed 192/192 queries

All queries done!


E0410 20:39:51.492724     746 chttp2_transport.cc:1385] unix:/tmp/tmp0uoy09bt_legal_ir.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [18]:
# Save vector DB retrieval results per split
with open("/kaggle/working/train_vdb_results.json", "w") as f:
    json.dump(train_vdb_results, f)
print("Saved train_vdb_results.json")

with open("/kaggle/working/val_vdb_results.json", "w") as f:
    json.dump(val_vdb_results, f)
print("Saved val_vdb_results.json")

with open("/kaggle/working/test_vdb_results.json", "w") as f:
    json.dump(test_vdb_results, f)
print("Saved test_vdb_results.json")

# Save filtered query texts (BM25 will need these as query strings)
all_queries = {}
all_queries.update(train_queries)
all_queries.update(val_queries)
all_queries.update(test_queries)

with open("/kaggle/working/all_queries.json", "w") as f:
    json.dump(all_queries, f)
print("Saved all_queries.json")

Saved train_vdb_results.json
Saved val_vdb_results.json
Saved test_vdb_results.json
Saved all_queries.json
